# Обучение модели на корпусе данных MIRACL

MIRACL (Multilingual Information Retrieval Across a Continuum of Languages) — это крупномасштабный многоязычный набор данных для оценки систем информационного поиска, охватывающий 18 языков и более трех миллиардов носителей. Корпус создан специально для задач одноязычного поиска (ad-hoc retrieval) и содержит тщательные аннотации релевантности от людей.

В данной работе была обучена модель multilingual-e5-small на корпусе данных MIRACL двумя стратегиями:
1) Baseline - стандартная форма обучения в 3 эпохи
2) Curriculum - обучение с варьированием сложности обучающих примеров

Импортирование библиотек:

In [1]:
import ir_datasets
import pandas as pd
import requests
from io import StringIO
from tqdm import tqdm
from collections import defaultdict
import builtins

In [2]:
import torch
import transformers
import sentence_transformers
from datasets import load_dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Transformers version: {transformers.__version__}")
print(f"Sentence-Transformers version: {sentence_transformers.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.7.1+cu118
CUDA available: True
Transformers version: 4.57.6
Sentence-Transformers version: 5.2.0
CUDA version: 11.8
GPU device: NVIDIA GeForce RTX 4060 Laptop GPU


Фиксируем кодировку для Windows:

In [3]:
_original_open = builtins.open
def utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, 
              newline=None, closefd=True, opener=None):
    if encoding is None and 'b' not in mode:
        encoding = 'utf-8'
    return _original_open(file, mode, buffering, encoding, errors, 
                         newline, closefd, opener)
builtins.open = utf8_open

In [4]:
queries_url = "https://huggingface.co/datasets/macavaney/miracl-noauth/resolve/main/miracl-v1.0-ru/topics/topics.miracl-v1.0-ru-train.tsv"

response = requests.get(queries_url)
response.encoding = 'utf-8'

queries_df = pd.read_csv(
    StringIO(response.text), 
    sep='\t', 
    names=['query_id', 'query'],
    encoding='utf-8'
)

queries_map = dict(zip(queries_df['query_id'], queries_df['query']))

print(f"✓ Train queries: {len(queries_map)}")
print(f"  Пример: {list(queries_map.values())[0]}")

✓ Train queries: 4683
  Пример: Когда был спущен на воду первый миноносец «Спокойный»?


Загружаем метки релевантности:

In [5]:
qrels_url = "https://huggingface.co/datasets/macavaney/miracl-noauth/resolve/main/miracl-v1.0-ru/qrels/qrels.miracl-v1.0-ru-train.tsv"

response = requests.get(qrels_url)
response.encoding = 'utf-8'

qrels_df = pd.read_csv(
    StringIO(response.text), 
    sep='\t', 
    names=['query_id', 'Q0', 'doc_id', 'relevance'],
    encoding='utf-8'
)

qrels_df = qrels_df[qrels_df['relevance'] > 0]

print(f" Qrels: {len(qrels_df)} релевантных пар")
print(f"  Уникальных запросов: {qrels_df['query_id'].nunique()}")

needed_docs = set(qrels_df['doc_id'].unique())
print(f"  Нужно загрузить: {len(needed_docs)} документов")

 Qrels: 10000 релевантных пар
  Уникальных запросов: 4683
  Нужно загрузить: 9108 документов


In [11]:
dataset = ir_datasets.load("miracl/ru")

corpus_map = {}
doc_count = 0

for doc in tqdm(dataset.docs_iter(), desc="Loading corpus"):
    try:
        if doc.doc_id in needed_docs:
            corpus_map[doc.doc_id] = {
                'title': doc.title if doc.title else "",
                'text': doc.text if doc.text else ""
            }
        
        doc_count += 1
        
        if len(corpus_map) >= len(needed_docs):
            print(f"\n  ✓ Нашли все {len(corpus_map)} нужных документов!")
            break
        if doc_count >= 500000:
            print(f"\n   Просмотрели {doc_count} документов, нашли {len(corpus_map)}")
            break        
            
    except (UnicodeDecodeError, AttributeError) as e:
        continue

print(f"\n✓ Загружено {len(corpus_map)} документов из corpus")
coverage = len(corpus_map) / len(needed_docs) * 100
print(f"  Покрытие: {coverage:.1f}% от нужных документов")

if corpus_map:
    first_doc = list(corpus_map.values())[0]
    print(f"  Пример: {first_doc['title'][:80]}...")

Loading corpus: 499999it [00:03, 144057.28it/s]


  ⚠️ Просмотрели 500000 документов, нашли 2396

✓ Загружено 2396 документов из corpus
  Покрытие: 26.3% от нужных документов
  Пример: Россия...


Создаем тренировочные пары:

In [12]:
print("\n[4/4] Создаём тренировочные пары")

train_examples = []

for idx, row in tqdm(qrels_df.iterrows(), total=len(qrels_df), desc="Создаем пары"):
    query_id = row['query_id']
    doc_id = row['doc_id']
    
    if query_id not in queries_map:
        continue
    if doc_id not in corpus_map:
        continue
    
    query_text = queries_map[query_id]
    doc = corpus_map[doc_id]
    passage_text = f"{doc['title']} {doc['text']}".strip()
    
    if len(passage_text) < 10:
        continue
    
    train_examples.append({
        'query_id': query_id,
        'query': query_text,
        'docid': doc_id,
        'passage': passage_text,
        'label': 1
    })

print(f"\n Создано {len(train_examples)} тренировочных пар")


[4/4] Создаём тренировочные пары


Создаем пары: 100%|███████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 70419.84it/s]


 Создано 2718 тренировочных пар


Создаем Dataframe для дальнейшей обработки:

In [15]:
df_train = pd.DataFrame(train_examples)

if len(df_train) > 0:
    df_train['query_len'] = df_train['query'].apply(lambda x: len(x.split()))
    df_train['passage_len'] = df_train['passage'].apply(lambda x: len(x.split()))
    
    print(f"\n Статистика:")
    print(f"  Примеров: {len(df_train)}")
    print(f"  Уникальных запросов: {df_train['query_id'].nunique()}")
    print(f"  Средний запрос: {df_train['query_len'].mean():.1f} слов")
    print(f"  Средний текст: {df_train['passage_len'].mean():.1f} слов")
    
    print("\nПример:")
    print(f"  Запрос: {df_train.iloc[0]['query']}")
    print(f"  Текст: {df_train.iloc[0]['passage'][:150]}...")


 Статистика:
  Примеров: 2718
  Уникальных запросов: 1810
  Средний запрос: 5.7 слов
  Средний текст: 87.9 слов

Пример:
  Запрос: Как звали предполагаемого убийцу Джона Кеннеди?
  Текст: Кеннеди, Джон Фицджералд 22 ноября 1963 года, совершая предвыборную поездку в Даллас, штат Техас, Джон Ф. Кеннеди был смертельно ранен из снайперской ...


Определяем функцию для вычисления сложности:

In [16]:
    def calculate_difficulty(row):
        query_words = set(row['query'].lower().split())
        passage_words = set(row['passage'].lower().split())
        
        intersection = len(query_words & passage_words)
        union = len(query_words | passage_words)
        lexical_overlap = intersection / union if union > 0 else 0
        
        norm_passage_len = min(row['passage_len'] / 500, 1.0)
        norm_query_len = min(row['query_len'] / 50, 1.0)
        
        difficulty = (norm_passage_len * 0.4 + 
                      norm_query_len * 0.2 + 
                      (1 - lexical_overlap) * 0.4)
        
        return difficulty, lexical_overlap
    
    df_train[['difficulty', 'lexical_overlap']] = df_train.apply(
        lambda row: pd.Series(calculate_difficulty(row)), axis=1
    )
    
    print(f"Сложность:")
    print(df_train['difficulty'].describe())

Сложность:
count    2718.000000
mean        0.481276
std         0.054931
min         0.187111
25%         0.447200
50%         0.471740
75%         0.506681
max         0.826351
Name: difficulty, dtype: float64


Разбиваем данные для эпох cirriculum learning

In [18]:
df_train_sorted = df_train.sort_values('difficulty').reset_index(drop=True)
    
n_stages = 4
stage_size = len(df_train_sorted) // n_stages
    
stages = {}
for i in range(n_stages):
    start_idx = i * stage_size
    end_idx = (i + 1) * stage_size if i < n_stages - 1 else len(df_train_sorted)
        
    stages[f'stage_{i+1}'] = df_train_sorted.iloc[start_idx:end_idx]
        
    print(f"\n Стадия {i+1}: {len(stages[f'stage_{i+1}'])} примеров")
    print(f"   Сложность: {stages[f'stage_{i+1}']['difficulty'].min():.3f} - "
            f"{stages[f'stage_{i+1}']['difficulty'].max():.3f}")
    print(f"   Средняя длина текста: {stages[f'stage_{i+1}']['passage_len'].mean():.1f}")
    print(f"   Средний Коэффициент Жаккара: {stages[f'stage_{i+1}']['lexical_overlap'].mean():.3f}")


 Стадия 1: 679 примеров
   Сложность: 0.187 - 0.447
   Средняя длина текста: 33.5
   Средний Коэффициент Жаккара: 0.059

 Стадия 2: 679 примеров
   Сложность: 0.447 - 0.472
   Средняя длина текста: 59.5
   Средний Коэффициент Жаккара: 0.025

 Стадия 3: 679 примеров
   Сложность: 0.472 - 0.507
   Средняя длина текста: 89.6
   Средний Коэффициент Жаккара: 0.019

 Стадия 4: 681 примеров
   Сложность: 0.507 - 0.826
   Средняя длина текста: 168.7
   Средний Коэффициент Жаккара: 0.014


Сохранение данных:

In [19]:
df_train_sorted.to_csv('miracl_ru_train_prepared.csv', index=False, encoding='utf-8')
print(f"✓ miracl_ru_train_prepared.csv ({len(df_train_sorted)} примеров)")
    
for stage_name, stage_df in stages.items():
    filename = f'miracl_ru_{stage_name}.csv'
    stage_df.to_csv(filename, index=False, encoding='utf-8')
    print(f"✓ {filename} ({len(stage_df)} примеров)")
    
print("\n ГОТОВО!")

✓ miracl_ru_train_prepared.csv (2718 примеров)
✓ miracl_ru_stage_1.csv (679 примеров)
✓ miracl_ru_stage_2.csv (679 примеров)
✓ miracl_ru_stage_3.csv (679 примеров)
✓ miracl_ru_stage_4.csv (681 примеров)

 ГОТОВО!


In [20]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader
from datetime import datetime

In [21]:
df_full = pd.read_csv('miracl_ru_train_prepared.csv', encoding='utf-8')

print(f" Загружено {len(df_full)} примеров")
print(f"  Уникальных запросов: {df_full['query_id'].nunique()}")

 Загружено 2718 примеров
  Уникальных запросов: 1810


Разделяем на train и validation

In [22]:
from sklearn.model_selection import train_test_split

unique_queries = df_full['query_id'].unique()
train_queries, val_queries = train_test_split(
    unique_queries, 
    test_size=0.2, 
    random_state=42
)

df_train = df_full[df_full['query_id'].isin(train_queries)]
df_val = df_full[df_full['query_id'].isin(val_queries)]

print(f"\n Разделение данных:")
print(f"  Train: {len(df_train)} примеров ({len(train_queries)} запросов)")
print(f"  Val: {len(df_val)} примеров ({len(val_queries)} запросов)")


 Разделение данных:
  Train: 2175 примеров (1448 запросов)
  Val: 543 примеров (362 запросов)


Загружаем модель:

In [23]:
model = SentenceTransformer('intfloat/multilingual-e5-small')

print(f" Модель загружена")
print(f"  Max seq length: {model.max_seq_length}")
print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"  Device: {model.device}")

 Модель загружена
  Max seq length: 512
  Embedding dim: 384
  Device: cuda:0


In [24]:
train_examples = []

for idx, row in df_train.iterrows():
    query_with_prefix = f"query: {row['query']}"
    passage_with_prefix = f"passage: {row['passage']}"
    
    train_examples.append(
        InputExample(texts=[query_with_prefix, passage_with_prefix])
    )

print(f"  Создано {len(train_examples)} InputExample")
print(f"  Пример query: {train_examples[0].texts[0][:80]}...")
print(f"  Пример passage: {train_examples[0].texts[1][:80]}...")

train_dataloader = DataLoader(
    train_examples, 
    shuffle=True,  
    batch_size=16  
)

print(f" DataLoader создан (batch_size=16)")

  Создано 2175 InputExample
  Пример query: query: В каком году город Львов был одним из четырёх городов Украины, принимавши...
  Пример passage: passage: Львов В 2012 году был одним из четырёх городов Украины, принимавших чем...
 DataLoader создан (batch_size=16)


Настраиваем функцию потерь:

In [25]:
train_loss = losses.MultipleNegativesRankingLoss(model)

In [26]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sample = df_val.sample(min(500, len(df_val)), random_state=42)

val_sentences1 = [f"query: {q}" for q in val_sample['query'].tolist()]
val_sentences2 = [f"passage: {p}" for p in val_sample['passage'].tolist()]
val_scores = [1.0] * len(val_sample)  # все пары релевантны

evaluator = EmbeddingSimilarityEvaluator(
    val_sentences1,
    val_sentences2,
    val_scores,
    name='miracl_ru_val'
)

print(f" Evaluator создан на {len(val_sample)} примерах")

 Evaluator создан на 500 примерах


In [28]:
import os
os.makedirs('models', exist_ok=True)

Обучение Baseline

In [29]:
num_epochs = 3  
warmup_steps = int(len(train_dataloader) * 0.1) 

print(f"\n  Параметры обучения:")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: 16")
print(f"  Steps per epoch: {len(train_dataloader)}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total training steps: {len(train_dataloader) * num_epochs}")

output_path = f'models/baseline_e5_small_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print(f"\n Модель будет сохранена в: {output_path}")
print("\n Начинаем обучение...\n")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluator=evaluator,
    evaluation_steps=len(train_dataloader) // 2,  # оценка 2 раза за эпоху
    output_path=output_path,
    save_best_model=True,
    show_progress_bar=True
)

print(" Обучение завершено")
print(f"\n Модель сохранена в: {output_path}")


  Параметры обучения:
  Epochs: 3
  Batch size: 16
  Steps per epoch: 136
  Warmup steps: 13
  Total training steps: 408

 Модель будет сохранена в: models/baseline_e5_small_20260123_142130

 Начинаем обучение...



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Miracl Ru Val Pearson Cosine,Miracl Ru Val Spearman Cosine
68,No log,No log,nan,nan
136,No log,No log,nan,nan
204,No log,No log,nan,nan
272,No log,No log,nan,nan
340,No log,No log,nan,nan
408,No log,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


 Обучение завершено

 Модель сохранена в: models/baseline_e5_small_20260123_142130


Теперь для Cirriculum learning:

In [30]:
stages_data = {}

for i in range(1, 5):
    filename = f'miracl_ru_stage_{i}.csv'
    
    if os.path.exists(filename):
        df = pd.read_csv(filename, encoding='utf-8')
        stages_data[f'stage_{i}'] = df
        print(f"  Stage {i}: {len(df)} примеров "
              f"(difficulty: {df['difficulty'].min():.3f}-{df['difficulty'].max():.3f})")
    else:
        print(f"  Файл {filename} не найден!")

print(f"\n Всего stages: {len(stages_data)}")
total_examples = sum(len(df) for df in stages_data.values())
print(f" Всего примеров: {total_examples}")

  Stage 1: 679 примеров (difficulty: 0.187-0.447)
  Stage 2: 679 примеров (difficulty: 0.447-0.472)
  Stage 3: 679 примеров (difficulty: 0.472-0.507)
  Stage 4: 681 примеров (difficulty: 0.507-0.826)

✓ Всего stages: 4
✓ Всего примеров: 2718


In [31]:
last_stage = stages_data['stage_4']
val_size = int(len(last_stage) * 0.1)

val_df = last_stage.sample(n=val_size, random_state=42)
stages_data['stage_4'] = last_stage.drop(val_df.index)

print(f" Validation: {len(val_df)} примеров")
print(f" Stage 4 после удаления val: {len(stages_data['stage_4'])} примеров")

# Создаём evaluator
val_sentences1 = [f"query: {q}" for q in val_df['query'].tolist()]
val_sentences2 = [f"passage: {p}" for p in val_df['passage'].tolist()]
val_scores = [1.0] * len(val_df)

evaluator = EmbeddingSimilarityEvaluator(
    val_sentences1[:500],  # ограничим для скорости
    val_sentences2[:500],
    val_scores[:500],
    name='curriculum_val'
)

print(f" Evaluator создан на {min(500, len(val_df))} примерах")

 Validation: 68 примеров
 Stage 4 после удаления val: 613 примеров
 Evaluator создан на 68 примерах


In [32]:
model_1 = SentenceTransformer('intfloat/multilingual-e5-small')

In [33]:
batch_size = 16
epochs_per_stage = 2 

train_loss = losses.MultipleNegativesRankingLoss(model)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_path = f'models/curriculum_e5_small_{timestamp}'
os.makedirs(base_output_path, exist_ok=True)

print(f"\n📋 Параметры:")
print(f"  Stages: {len(stages_data)}")
print(f"  Epochs per stage: {epochs_per_stage}")
print(f"  Batch size: {batch_size}")
print(f"  Base path: {base_output_path}")

training_history = []


📋 Параметры:
  Stages: 4
  Epochs per stage: 2
  Batch size: 16
  Base path: models/curriculum_e5_small_20260123_143310


Обучение по стадиям:

In [34]:
for stage_idx, (stage_name, stage_df) in enumerate(stages_data.items(), 1):
    
    print(f" Стадия {stage_idx}/4: {stage_name.upper()}")
    
    print(f"  Примеров: {len(stage_df)}")
    print(f"  Сложность: {stage_df['difficulty'].min():.3f} - {stage_df['difficulty'].max():.3f}")
    print(f"  Средняя длина примера: {stage_df['passage_len'].mean():.1f} слов")
    
    stage_examples = []
    
    for idx, row in stage_df.iterrows():
        query_with_prefix = f"query: {row['query']}"
        passage_with_prefix = f"passage: {row['passage']}"
        
        stage_examples.append(
            InputExample(texts=[query_with_prefix, passage_with_prefix])
        )
    
    stage_dataloader = DataLoader(
        stage_examples,
        shuffle=True,
        batch_size=batch_size
    )
    
    steps_per_epoch = len(stage_dataloader)
    total_steps = steps_per_epoch * epochs_per_stage
    warmup_steps = int(total_steps * 0.1)
    
    print(f"\n  Steps per epoch: {steps_per_epoch}")
    print(f"  Total steps: {total_steps}")
    print(f"  Warmup steps: {warmup_steps}")
    
    stage_output_path = os.path.join(base_output_path, f'stage_{stage_idx}')
    
    print(f"\n   Начинаем обучение Stage {stage_idx}...\n")
    
    model.fit(
        train_objectives=[(stage_dataloader, train_loss)],
        epochs=epochs_per_stage,
        warmup_steps=warmup_steps,
        evaluator=evaluator,
        evaluation_steps=max(steps_per_epoch // 2, 1),  # оценка 2 раза за эпоху
        output_path=stage_output_path,
        save_best_model=True,
        show_progress_bar=True,
        checkpoint_save_steps=steps_per_epoch,  # чекпоинт каждую эпоху
        checkpoint_path=stage_output_path,
        checkpoint_save_total_limit=2
    )
    
    print(f"\n  ✓ Stage {stage_idx} завершена!")
    print(f"   Сохранено в: {stage_output_path}")
    
    training_history.append({
        'stage': stage_idx,
        'stage_name': stage_name,
        'num_examples': len(stage_df),
        'epochs': epochs_per_stage,
        'total_steps': total_steps,
        'output_path': stage_output_path
    })


print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")

final_model_path = os.path.join(base_output_path, 'final_model')
model.save(final_model_path)
print(f"✓ Финальная модель: {final_model_path}")

import json

history_path = os.path.join(base_output_path, 'training_history.json')
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2, ensure_ascii=False)

print(f"  История обучения: {history_path}")

print(" CURRICULUM LEARNING ЗАВЕРШЕНО!")

print(f"\n  Итоги:")
print(f"  Обучено stages: {len(training_history)}")
print(f"  Всего примеров: {sum(h['num_examples'] for h in training_history)}")
print(f"  Всего эпох: {sum(h['epochs'] for h in training_history)}")
print(f"  Всего steps: {sum(h['total_steps'] for h in training_history)}")

print(f"\n  Структура сохранённых моделей:")
for h in training_history:
    print(f"  Stage {h['stage']}: {h['output_path']}")
print(f"  Final: {final_model_path}")

comparison_info = {
    'curriculum_model_path': final_model_path,
    'total_examples': sum(h['num_examples'] for h in training_history),
    'training_stages': len(training_history),
    'epochs_per_stage': epochs_per_stage,
    'timestamp': timestamp
}

with open(os.path.join(base_output_path, 'model_info.json'), 'w') as f:
    json.dump(comparison_info, f, indent=2)

 Стадия 1/4: STAGE_1
  Примеров: 679
  Сложность: 0.187 - 0.447
  Средняя длина примера: 33.5 слов

  Steps per epoch: 43
  Total steps: 86
  Warmup steps: 8

   Начинаем обучение Stage 1...



Step,Training Loss,Validation Loss,Curriculum Val Pearson Cosine,Curriculum Val Spearman Cosine
21,No log,No log,nan,nan
42,No log,No log,nan,nan
43,No log,No log,nan,nan
63,No log,No log,nan,nan
84,No log,No log,nan,nan
86,No log,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.p


  ✓ Stage 1 завершена!
   Сохранено в: models/curriculum_e5_small_20260123_143310\stage_1
 Стадия 2/4: STAGE_2
  Примеров: 679
  Сложность: 0.447 - 0.472
  Средняя длина примера: 59.5 слов

  Steps per epoch: 43
  Total steps: 86
  Warmup steps: 8

   Начинаем обучение Stage 2...



Step,Training Loss,Validation Loss,Curriculum Val Pearson Cosine,Curriculum Val Spearman Cosine
21,No log,No log,nan,nan
42,No log,No log,nan,nan
43,No log,No log,nan,nan
63,No log,No log,nan,nan
84,No log,No log,nan,nan
86,No log,No log,nan,nan


...

C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.p


  ✓ Stage 3 завершена!
   Сохранено в: models/curriculum_e5_small_20260123_143310\stage_3
 Стадия 4/4: STAGE_4
  Примеров: 613
  Сложность: 0.507 - 0.826
  Средняя длина примера: 167.7 слов

  Steps per epoch: 39
  Total steps: 78
  Warmup steps: 7

   Начинаем обучение Stage 4...



Step,Training Loss,Validation Loss,Curriculum Val Pearson Cosine,Curriculum Val Spearman Cosine
19,No log,No log,nan,nan
38,No log,No log,nan,nan
39,No log,No log,nan,nan
57,No log,No log,nan,nan
76,No log,No log,nan,nan
78,No log,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.p


  ✓ Stage 4 завершена!
   Сохранено в: models/curriculum_e5_small_20260123_143310\stage_4
СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
✓ Финальная модель: models/curriculum_e5_small_20260123_143310\final_model
  История обучения: models/curriculum_e5_small_20260123_143310\training_history.json
 CURRICULUM LEARNING ЗАВЕРШЕНО!

  Итоги:
  Обучено stages: 4
  Всего примеров: 2650
  Всего эпох: 8
  Всего steps: 336

  Структура сохранённых моделей:
  Stage 1: models/curriculum_e5_small_20260123_143310\stage_1
  Stage 2: models/curriculum_e5_small_20260123_143310\stage_2
  Stage 3: models/curriculum_e5_small_20260123_143310\stage_3
  Stage 4: models/curriculum_e5_small_20260123_143310\stage_4
  Final: models/curriculum_e5_small_20260123_143310\final_model
